# 🗓️ 30일차 스터디 노트북 — 커서를 이용한 연결 리스트

**오늘 범위**: 08-3 커서를 이용한 연결 리스트 — 커서(cursor) 개념 · 배열 안에 노드 저장 · **프리 리스트(free list)** · 실습 8-3 `ArrayLinkedList` · 보충수업 8-3 파이썬 논리 연산자

## 난이도 태그
🟢 **기본** (전원 필수) / 🟡 **표준** (팀 목표선) / 🔴 **심화** (도전)

## 유형 태그
**[표]** 배열 상태 직접 채우기 · **[손]** · **[빈칸]** · **[예측]** · **[구현]** · **[디버깅]** · **[설명]** · **[실험]**

---

## 어제와 무엇이 달라지나

29일차에서 배운 08-2는 **노드를 객체로** 만들었어. 오늘은 **배열 안의 원소로** 만들어.

교재 342p가 그 동기를 이렇게 말해:
> **"08-2절에서 학습한 포인터를 이용한 연결 리스트는 '노드를 삽입·삭제할 때 데이터를 이동하지 않고 처리'하는 특징이 있습니다. 하지만 노드를 삽입·삭제할 때마다 내부에서 노드용 인스턴스를 생성하고 소멸합니다. 이때 메모리를 확보하고 해제하는 데 쓰는 비용을 결코 무시할 수 없습니다."**

| | 08-2 (29일차) | 08-3 (오늘) |
|---|---|---|
| 노드의 정체 | `Node` **객체** | 배열 `n[]` 의 **원소** |
| 뒤쪽 포인터 | 객체 **참조** | **인덱스**(정수) = **커서** |
| "없음" 표시 | `None` | `Null = -1` |
| 노드 생성 | 매번 `Node()` 인스턴스 생성 | **미리 만들어둔 배열 재사용** |
| 리스트 끝 판정 | `ptr is None` | `ptr == Null` |

교재 343p: **"int형 정숫값인 인덱스로 나타낸 포인터를 커서(cursor)라고 합니다."**

## 오늘 반드시 잡아야 할 두 가지

> **① 배열의 물리적 위치 ≠ 리스트의 논리적 순서** 🔥
> 교재 349p: **"배열에서 물리적인 위치 관계와 연결 리스트의 논리적인 순서 관계가 일치하는 것은 아닙니다."**
>
> **② 프리 리스트(free list) — 삭제된 레코드를 관리하는 두 번째 연결 리스트**
> 하나의 배열 안에 **두 개의 연결 리스트**가 동시에 존재해!

## 진행 순서
**커서 개념(1~3) → 배열↔리스트 대응(4~6) → 프리 리스트(7~11) → 코드 구현(12~16) → 함정과 성능(17~20)**

> 📁 아래 **"부록"** 셀을 **가장 먼저 실행**해.

---

## 📁 부록 — 오늘의 도구 (제일 먼저 실행!)

In [ ]:
from __future__ import annotations
from typing import Any
import time, sys

Null = -1

class Node:
    """연결 리스트용 노드 클래스(배열 커서 버전)"""
    def __init__(self, data=Null, next=Null, dnext=Null):
        self.data = data      # 데이터
        self.next = next      # 리스트의 뒤쪽 포인터(커서)
        self.dnext = dnext    # 프리 리스트의 뒤쪽 포인터(커서)


class ArrayLinkedList:
    """연결 리스트 클래스(배열 커서 버전)"""
    def __init__(self, capacity: int):
        self.head = Null            # 머리 노드
        self.current = Null         # 주목 노드
        self.max = Null             # 사용 중인 꼬리 레코드
        self.deleted = Null         # 프리 리스트의 머리 노드
        self.capacity = capacity    # 리스트의 크기
        self.n = [Node()] * self.capacity   # 리스트 본체 (⚠️ 17번!)
        self.no = 0

    def __len__(self) -> int:
        return self.no

    def get_insert_index(self):
        """다음에 삽입할 레코드의 인덱스를 구함"""
        if self.deleted == Null:                # 삭제 레코드가 없음
            if self.max + 1 < self.capacity:
                self.max += 1
                return self.max                 # 새 레코드를 사용
            else:
                return Null                     # 크기 초과
        else:
            rec = self.deleted
            self.deleted = self.n[rec].dnext    # 프리 리스트에서 맨 앞을 꺼냄
            return rec

    def delete_index(self, idx: int) -> None:
        """레코드 idx를 프리 리스트에 등록"""
        if self.deleted == Null:
            self.deleted = idx
            self.n[idx].dnext = Null
        else:
            rec = self.deleted
            self.deleted = idx                  # idx를 프리 리스트 맨 앞에 삽입
            self.n[idx].dnext = rec

    def search(self, data: Any) -> int:
        cnt = 0
        ptr = self.head
        while ptr != Null:
            if self.n[ptr].data == data:
                self.current = ptr
                return cnt
            cnt += 1
            ptr = self.n[ptr].next
        return Null

    def __contains__(self, data: Any) -> bool:
        return self.search(data) >= 0

    def add_first(self, data: Any) -> None:
        ptr = self.head
        rec = self.get_insert_index()
        if rec != Null:
            self.head = self.current = rec
            self.n[self.head] = Node(data, ptr)
            self.no += 1

    def add_last(self, data: Any) -> None:
        if self.head == Null:
            self.add_first(data)
        else:
            ptr = self.head
            while self.n[ptr].next != Null:
                ptr = self.n[ptr].next
            rec = self.get_insert_index()
            if rec != Null:
                self.n[ptr].next = self.current = rec
                self.n[rec] = Node(data)
                self.no += 1

    def remove_first(self) -> None:
        if self.head != Null:
            ptr = self.n[self.head].next
            self.delete_index(self.head)
            self.head = self.current = ptr
            self.no -= 1

    def remove_last(self) -> None:
        if self.head != Null:
            if self.n[self.head].next == Null:
                self.remove_first()
            else:
                ptr = self.head
                pre = self.head
                while self.n[ptr].next != Null:
                    pre = ptr
                    ptr = self.n[ptr].next
                self.n[pre].next = Null
                self.delete_index(ptr)
                self.current = pre
                self.no -= 1

    def remove(self, p: int) -> None:
        if self.head != Null:
            if p == self.head:
                self.remove_first()
            else:
                ptr = self.head
                while self.n[ptr].next != p:
                    ptr = self.n[ptr].next
                    if ptr == Null:
                        return
                self.n[ptr].next = Null          # ← 18번에서 다룰 줄
                self.delete_index(p)
                self.n[ptr].next = self.n[p].next
                self.current = ptr
                self.no -= 1

    def remove_current_node(self) -> None:
        self.remove(self.current)

    def clear(self) -> None:
        while self.head != Null:
            self.remove_first()
        self.current = Null

    def next(self) -> bool:
        if self.current == Null or self.n[self.current].next == Null:
            return False
        self.current = self.n[self.current].next
        return True

    def print_current_node(self) -> None:
        if self.current == Null:
            print('주목 노드가 없습니다.')
        else:
            print(self.n[self.current].data)

    def print(self) -> None:
        ptr = self.head
        while ptr != Null:
            print(self.n[ptr].data)
            ptr = self.n[ptr].next

    def dump(self) -> None:
        for i in range(self.capacity):
            nd = self.n[i]
            print(f'[{i}] {nd.data} {nd.next} {nd.dnext}')

    def __iter__(self):
        return ArrayLinkedListIterator(self.n, self.head)


class ArrayLinkedListIterator:
    def __init__(self, n, head: int):
        self.n = n
        self.current = head
    def __iter__(self):
        return self
    def __next__(self) -> Any:
        if self.current == Null:
            raise StopIteration
        data = self.n[self.current].data
        self.current = self.n[self.current].next
        return data


# ===== 실험용 시각화 =====
def show(l, title="", upto=None):
    """리스트 순서 + 배열 상태 + 프리 리스트를 한눈에"""
    if title: print(title)
    order, p = [], l.head
    while p != Null:
        mark = "*" if p is not None and p == l.current else ""
        order.append(f"{l.n[p].data}({p}){mark}")
        p = l.n[p].next
    free, p = [], l.deleted
    while p != Null:
        free.append(str(p)); p = l.n[p].dnext
    print(f"  리스트 : {' → '.join(order) if order else '(빈 리스트)'}")
    end = l.capacity if upto is None else min(upto, l.capacity)
    print(f"  {'idx':>4} | {'data':>6} | {'next':>4} | {'dnext':>5}")
    print("  " + "-"*32)
    for i in range(end):
        nd = l.n[i]
        used = any(x.startswith(f"{nd.data}({i})") for x in order)
        tag = "◀리스트" if used else ("◀프리" if str(i) in free else "")
        print(f"  {i:>4} | {str(nd.data):>6} | {nd.next:>4} | {nd.dnext:>5}  {tag}")
    print(f"  head={l.head}  current={l.current}  max={l.max}  deleted={l.deleted}  no={l.no}")
    print(f"  프리 리스트: {' → '.join(free) if free else '(없음)'}\n")

def make(cap, *vals):
    l = ArrayLinkedList(cap)
    for v in vals: l.add_last(v)
    return l

print("준비 완료 ✅\n")
show(make(8, 'A','B','C','D','E'), "예시: A~E를 add_last로 넣은 상태", upto=6)

---
# 🔁 [Remind] 워밍업 — 29일차 되감기

오늘 코드는 **어제 코드와 거의 1:1 대응**이야. 교재 349p도 이렇게 말해:
> **"실습 8-3 프로그램은 08-2절에서 작성한 실습 8-1 프로그램의 연결 리스트 LinkedList 클래스와 함수에 거의 일대일로 대응합니다."**

### R-1. 🟢 [설명] 번역표 만들기

어제 코드를 오늘 코드로 **기계적으로 번역**해봐.

| 08-2 (29일차) | 08-3 (오늘) |
|---|---|
| `ptr = self.head` | ① |
| `ptr is None` | ② |
| `ptr = ptr.next` | ③ |
| `ptr.data` | ④ |
| `self.head = None` | ⑤ |
| `Node(data, ptr)` (새 객체 생성) | ⑥ |

- 규칙 하나만 잡으면 돼: **"노드를 가리키던 참조가 인덱스가 되고, `노드.필드` 는 `n[인덱스].필드` 가 된다."**
- `None` 이 `Null(-1)` 로 바뀌는데, 왜 하필 **-1**일까? 0이면 안 될까? 🔥

### R-2. 🟡 [설명] 어제의 두 가지 약점

29일차 21~22번에서 확인한 연결 리스트의 약점 두 개를 적어봐.
- 약점 1 (메모리): ①________________
- 약점 2 (`add_last`): ②________________

- 오늘의 커서 방식은 이 중 **어느 쪽**을 해결할까? 그리고 **어느 쪽은 그대로**일까?
- 예측만 해두고 20번에서 확인해.

*(여기에 답 작성)*

---
# 🎯 PART 1 — 커서란 무엇인가 (1~3번)

### 1. 🟢 [설명] 참조 대신 인덱스

교재 343p [그림 8-16]. 같은 연결 리스트를 두 가지로 그린 거야.

**ⓐ 논리적 이미지**
```
head → [A] → [B] → [C] → [D] → [E] → [F] → None
```

**ⓑ 배열로 구현**
```
head = 1
      idx | data | next
       0  |  D   |  2
       1  |  A   |  4
       2  |  E   |  5
       3  |  C   |  0
       4  |  B   |  3
       5  |  F   | -1
       6  |  -   |  -
       7  |  -   |  -
```

**직접 따라가 봐:**
- `head = 1` 이니 머리 노드는 `n[1]` = ①____
- `n[1].next = 4` → 다음은 `n[4]` = ②____
- `n[4].next = 3` → 다음은 `n[3]` = ③____
- `n[3].next = 0` → ④____
- `n[0].next = 2` → ⑤____
- `n[2].next = 5` → ⑥____
- `n[5].next = -1` → ⑦________________

- 교재 343p: **"꼬리 노드의 뒤쪽 커서는 -1입니다."**
  → 왜 `-1` 이 "끝"을 뜻하기에 적합할까? **배열 인덱스로 절대 나올 수 없는 값**이라는 게 핵심이야.
- 🔥 **배열에 담긴 순서(D, A, E, C, B, F)와 리스트 순서(A, B, C, D, E, F)가 완전히 달라.** 이게 오늘 가장 중요한 개념이야.

*(답을 적은 뒤 실행)*

In [ ]:
# 교재 그림 8-16의 배열 상태를 직접 만들어보기
demo = ArrayLinkedList(8)
demo.n = [Node() for _ in range(8)]      # (17번에서 다룰 이유로 리스트 내포 사용)
table = [('D', 2), ('A', 4), ('E', 5), ('C', 0), ('B', 3), ('F', -1)]
for i, (d, nx) in enumerate(table):
    demo.n[i] = Node(d, nx)
demo.head = 1; demo.max = 5; demo.no = 6

print("배열에 담긴 순서:", [demo.n[i].data for i in range(6)])
print("\n커서를 따라가면:")
p, order = demo.head, []
while p != Null:
    print(f"  n[{p}] = '{demo.n[p].data}',  next = {demo.n[p].next}")
    order.append(demo.n[p].data)
    p = demo.n[p].next
print(f"\n리스트 순서: {' → '.join(order)}")
print(f"\n🔥 배열 순서 {[demo.n[i].data for i in range(6)]} ≠ 리스트 순서 {order}")

### 2. 🟢 [설명] `Null = -1` 의 의미

```python
Null = -1

class Node:
    def __init__(self, data=Null, next=Null, dnext=Null):
```

- `Null` 은 어제의 무엇에 해당해? ①____
- **세 곳**에서 쓰여. 각각 무슨 뜻인지 적어봐:
  - `head == Null` → ②________________
  - `n[p].next == Null` → ③________________
  - `deleted == Null` → ④________________ (7번에서)

🔥 **파이썬 함정 하나**: `Null` 이 `-1` 인데, 파이썬에서 `n[-1]` 은 **에러가 아니라 마지막 원소**야!
- 그럼 실수로 `self.n[Null]` 을 읽으면? **에러 없이 엉뚱한 값**이 나와.
- 이게 왜 위험할까? 25일차 4번(도수 정렬의 음수 인덱스), 29일차 4번과 같은 계열이야.
- 코드가 이걸 어떻게 막고 있지? (힌트: `while ptr != Null:` 을 **먼저** 검사)

*(답을 적은 뒤 실행)*

In [ ]:
print(f"Null = {Null}")
l = make(5, 'A', 'B', 'C')
print(f"\n리스트: A → B → C")
print(f"  n[Null] = n[-1] → data = {l.n[Null].data!r}   ← 에러가 안 난다! 🔥")
print(f"  파이썬 음수 인덱스는 '뒤에서부터'를 뜻하기 때문")

print("\n[코드는 어떻게 막고 있나]")
print("  while ptr != Null:          ← 먼저 검사")
print("      ... self.n[ptr] ...     ← 검사를 통과해야 접근")
print("  → '검사 후 접근' 순서가 지켜져야만 안전하다\n")

print("[만약 순서가 뒤바뀌면?]")
p = Null
print(f"  ptr = Null 인데 self.n[ptr].next 를 읽으면 → {l.n[p].next}")
print("  → 존재하지 않는 노드의 커서를 읽어 리스트가 조용히 망가진다")

### 3. 🟡 [설명] 필드가 왜 이렇게 많아졌나

```python
class ArrayLinkedList:
    def __init__(self, capacity):
        self.head = Null        # ①
        self.current = Null     # ②
        self.max = Null         # ③  ← 새로 생김!
        self.deleted = Null     # ④  ← 새로 생김!
        self.capacity = capacity
        self.n = [Node()] * capacity
        self.no = 0
```

어제는 `no`, `head`, `current` **3개**뿐이었는데 오늘은 **6개**야.

**표를 채워봐:**

| 필드 | 무엇을 담나 | 어제(08-2)에 있었나 |
|---|---|---|
| `head` | ① | ✅ |
| `current` | ② | ✅ |
| `no` | ③ | ✅ |
| `capacity` | ④ | ❌ |
| `max` | ⑤ | ❌ |
| `deleted` | ⑥ | ❌ |

- 교재 351p: **"max: 배열에서 맨 끝 쪽에 저장되는 노드의 레코드 번호입니다."**
  → `max` 가 필요한 이유는? (힌트: 다음에 **새 레코드**를 어디에 만들지 알아야 하니까)
- `deleted` 는 7번에서 본격적으로 다뤄.
- `Node` 에도 필드가 하나 늘었어: `dnext`. 교재 351p: **"dnext: 프리 리스트의 뒤쪽 포인터"**
- 🔥 **왜 `next` 하나로 안 되고 `dnext` 가 따로 필요할까?** 예측해봐. (7번에서 답이 나와)

*(답을 적은 뒤 실행)*

In [ ]:
l = ArrayLinkedList(6)
print("빈 리스트의 필드")
print(f"  head={l.head}  current={l.current}  max={l.max}  deleted={l.deleted}  no={l.no}  capacity={l.capacity}")
print("  → head, current, max, deleted 모두 Null(-1)로 시작\n")

for v in ['A','B','C']:
    l.add_last(v)
    print(f"add_last('{v}') → max={l.max}, no={l.no}, head={l.head}")
print("\n  → max는 '지금까지 써본 가장 큰 레코드 번호'")
print("  → 다음 새 레코드는 max+1 번에 만들어진다")

---
# 📊 PART 2 — 배열 상태를 직접 채우기 (4~6번)

> **오늘의 기본기.** 커서를 따라가며 표를 채우는 연습.

### 4. 🟢 [표] 삽입 과정 추적 — 교재 그림 8-17

교재 344p [그림 8-17]. 1번 문제의 리스트 맨 앞에 노드 `G`를 삽입해.

**삽입 전**
```
head = 1
 idx | data | next
  0  |  D   |  2
  1  |  A   |  4
  2  |  E   |  5
  3  |  C   |  0
  4  |  B   |  3
  5  |  F   | -1
  6  |  -   |  -
  7  |  -   |  -
```

**`add_first('G')` 를 실행하면:**

```python
ptr = self.head              # ①  ptr = ____
rec = self.get_insert_index()  # ②  rec = ____ (프리 리스트가 비었으니 max+1)
if rec != Null:
    self.head = self.current = rec       # ③  head = ____
    self.n[self.head] = Node(data, ptr)  # ④  n[____] = Node('G', ____)
    self.no += 1
```

**삽입 후 표를 채워봐:**
```
head = ⑤____
 idx | data | next
  0  |  D   |  2
  1  |  A   |  4
  2  |  E   |  5
  3  |  C   |  0
  4  |  B   |  3
  5  |  F   | -1
  6  | ⑥__  | ⑦__
  7  |  -   |  -
```

- 교재 344p: **"head를 1에서 6으로 업데이트하고 노드 G의 뒤쪽 커서를 1로 했을 뿐입니다."**
- 🔥 **기존 원소가 하나도 안 움직였지?** 배열인데 왜 밀리지 않았을까? 29일차 1번의 배열 삽입과 비교해봐.
- `max` 는 어떻게 변했어? ⑧____

*(답을 적은 뒤 실행)*

In [ ]:
demo = ArrayLinkedList(8)
demo.n = [Node() for _ in range(8)]
for i, (d, nx) in enumerate([('D',2),('A',4),('E',5),('C',0),('B',3),('F',-1)]):
    demo.n[i] = Node(d, nx)
demo.head = 1; demo.max = 5; demo.no = 6

print("삽입 전")
show(demo, upto=8)

demo.add_first('G')
print("add_first('G') 후")
show(demo, upto=8)
print("→ 배열의 0~5번은 값도 next도 하나도 안 바뀌었다! 🔥")

### 5. 🟡 [표] `add_last` 추적

```python
if self.head == Null:
    self.add_first(data)
else:
    ptr = self.head
    while self.n[ptr].next != Null:      # ① 꼬리 찾기
        ptr = self.n[ptr].next
    rec = self.get_insert_index()
    if rec != Null:
        self.n[ptr].next = self.current = rec   # ②
        self.n[rec] = Node(data)                # ③
        self.no += 1
```

빈 리스트에 `add_last` 로 `A, B, C` 를 차례로 넣어봐.

| 호출 | head | max | 배열 상태 (idx: data/next) |
|---|---|---|---|
| 시작 | -1 | -1 | (전부 비어 있음) |
| `add_last('A')` | ① | ② | ③ |
| `add_last('B')` | ④ | ⑤ | ⑥ |
| `add_last('C')` | ⑦ | ⑧ | ⑨ |

- 🔥 `self.n[rec] = Node(data)` 에서 `next` 를 안 넘겼어. 그럼 기본값이 뭐지? ⑩____
  → 그게 왜 맞는 값이야?
- ①의 `while` 은 29일차 8번과 완전히 같은 구조야. 시간 복잡도는? ⑪____
- **빈 리스트일 때 `add_first` 로 넘기는 이유**는 어제와 같아. 뭐였지?

*(답을 적은 뒤 실행)*

In [ ]:
l = ArrayLinkedList(6)
print("시작"); show(l, upto=3)
for v in ['A','B','C']:
    l.add_last(v)
    print(f"add_last('{v}') 후")
    show(l, upto=3)
print("→ add_last는 배열 순서와 리스트 순서가 우연히 일치한다 (아직 삭제가 없었으니까)")

### 6. 🔴 [표] 🔥 배열 순서 ≠ 리스트 순서를 직접 만들어보기

교재 349p의 핵심 문장:
> **"삽입된 노드의 저장 장소는 '배열 안에서 가장 끝쪽에 있는 인덱스의 위치'입니다. 연결 리스트에서 맨 끝이 아닌 것에 주의하세요. 배열에서 물리적인 위치 관계와 연결 리스트의 논리적인 순서 관계가 일치하는 것은 아닙니다."**

**직접 예측해봐.** 다음 순서로 조작하면 배열과 리스트가 각각 어떻게 될까?

```python
l = ArrayLinkedList(8)
l.add_last('A')      # 레코드 ①____ 에 저장
l.add_last('B')      # 레코드 ②____
l.add_last('C')      # 레코드 ③____
l.add_first('X')     # 레코드 ④____ 🔥
l.add_first('Y')     # 레코드 ⑤____
```

- 최종 **리스트 순서**: ⑥________________
- 최종 **배열 순서**(인덱스 0부터): ⑦________________
- 두 개가 다르지? `add_first` 인데 배열의 **뒤쪽**에 저장되는 이유를 설명해봐. 🔥

*(예측을 적은 뒤 실행)*

In [ ]:
l = ArrayLinkedList(8)
for op, v in [('last','A'), ('last','B'), ('last','C'), ('first','X'), ('first','Y')]:
    if op == 'last': l.add_last(v)
    else: l.add_first(v)
    order, p = [], l.head
    while p != Null:
        order.append(l.n[p].data); p = l.n[p].next
    arr = [l.n[i].data for i in range(l.max+1)]
    print(f"add_{op}('{v}') → 리스트 {order}, 배열 {arr}")

print()
show(l, upto=6)
print("🔥 add_first('Y')인데 배열의 마지막(4번)에 저장됐다!")
print("   '리스트의 맨 앞'과 '배열의 맨 앞'은 아무 관계가 없다")

---
# ♻️ PART 3 — 프리 리스트 (7~11번)

> **오늘의 하이라이트.** 교재 350~352p.
>
> 🔥 **하나의 배열 안에 연결 리스트가 두 개** 들어 있어:
> - **데이터 리스트**: `head` 에서 시작, `next` 커서로 연결 → **실제 데이터의 순서**
> - **프리 리스트**: `deleted` 에서 시작, `dnext` 커서로 연결 → **삭제되어 비어 있는 레코드들**

### 7. 🟢 [설명] 왜 프리 리스트가 필요한가

교재 350p가 문제를 이렇게 제기해:
> **"만약 삭제를 여러 번 반복하면 배열 안에 빈 레코드가 많이 생깁니다. 삭제되는 레코드가 딱 하나라면 그 인덱스를 어떤 변수에 넣어 놓고 관리함으로써 쉽게 재사용할 수 있습니다. 그러나 실제로는 여러 레코드가 삭제되므로 그렇게 간단하지 않습니다."**

**상황을 상상해봐**: capacity=8인 배열에서 레코드 1, 3, 5를 삭제했어.

- 다음에 삽입할 때 **어디에** 넣어야 할까? 후보가 ①____개나 있어.
- 이 후보들을 어떻게 기억할까? 방법을 3개 떠올려봐:
  - 방법 A: 매번 배열 전체를 훑어 빈 곳을 찾는다 → 시간 복잡도 ②____
  - 방법 B: 빈 레코드 목록을 별도 리스트로 관리 → 문제점: ③________________
  - 방법 C: 🔥 **빈 레코드들끼리 서로 연결한다** → 추가 메모리 ④____

- 교재 351p: **"연결 리스트인 프리 리스트(free list)는 삭제된 레코드 그룹을 관리할 때 사용하는 자료구조입니다."**
- 방법 C가 천재적인 이유: **빈 레코드는 어차피 안 쓰이니까, 그 안의 `dnext` 필드를 공짜로 빌려 쓸 수 있어!** 추가 배열이 전혀 필요 없지.
- 그래서 `Node` 에 `dnext` 필드가 있는 거야 (3번의 마지막 질문 답).

*(답을 적은 뒤 실행)*

In [ ]:
l = make(8, 'A','B','C','D','E')
print("A~E 삽입 직후 — 프리 리스트는 비어 있다")
show(l, upto=6)

l.remove(1)
print("레코드 1(B) 삭제")
show(l, upto=6)
print("→ n[1]의 dnext가 프리 리스트를 만드는 데 재사용된다 🔥")

### 8. 🔴 [손] 🔥 프리 리스트는 **스택**이다

`delete_index` 와 `get_insert_index` 를 나란히 놓고 봐.

```python
def delete_index(self, idx):        # 삭제 → 프리 리스트에 등록
    if self.deleted == Null:
        self.deleted = idx
        self.n[idx].dnext = Null
    else:
        rec = self.deleted
        self.deleted = idx          # ← idx를 맨 앞에!
        self.n[idx].dnext = rec

def get_insert_index(self):         # 삽입 → 프리 리스트에서 꺼냄
    if self.deleted == Null:
        ... (새 레코드 사용)
    else:
        rec = self.deleted
        self.deleted = self.n[rec].dnext   # ← 맨 앞을 꺼냄
        return rec
```

- 등록은 **맨 ①____** 에, 꺼내기도 **맨 ②____** 에서.
- 그럼 **가장 최근에 삭제된 레코드**가 **가장 먼저** 재사용돼. 이 성질을 뭐라고 부르지? ③____
- 🔥 이 자료구조 4일차에 배웠지? ④____ 야!

**직접 예측해봐:**
```
레코드 1, 3, 5 순서로 삭제 → 프리 리스트: ⑤________________
그다음 add_first를 3번 하면 각각 레코드 ⑥__, ⑦__, ⑧__ 에 저장
```

- 교재 351p [그림 8-19] ⓐ의 설명: **"3개의 레코드 1, 3, 5가 삭제되어 프리 리스트는 3 → 1 → 5가 됩니다."**
  → 삭제 순서가 1, 3, 5인데 프리 리스트가 **3 → 1 → 5** 인 게 이상하지 않아? 순서를 다시 확인해봐. 🔥

*(예측을 적은 뒤 실행)*

In [ ]:
l = make(8, 'A','B','C','D','E','F')
print("A~F 삽입 (레코드 0~5 사용)")
show(l, upto=6)

for rec in (1, 3, 5):
    l.remove(rec)
    free, p = [], l.deleted
    while p != Null: free.append(str(p)); p = l.n[p].dnext
    print(f"레코드 {rec} 삭제 → 프리 리스트: {' → '.join(free)}")

print("\n→ 삭제 순서 1, 3, 5 인데 프리 리스트는 5 → 3 → 1")
print("   (가장 최근에 삭제된 5가 맨 앞) = LIFO = 스택 🔥\n")

print("이제 삽입하면 어느 레코드를 쓸까?")
for v in ['X','Y','Z','W']:
    l.add_first(v)
    p = l.head
    free, q = [], l.deleted
    while q != Null: free.append(str(q)); q = l.n[q].dnext
    print(f"  add_first('{v}') → 레코드 {p} 사용, 남은 프리: {' → '.join(free) if free else '(없음)'}")
print("\n→ 5, 3, 1 순으로 재사용 (LIFO), 다 쓰면 max+1 로 새 레코드 ✅")

### 9. 🟡 [표] 교재 그림 8-19 완전 추적

교재 352p [그림 8-19]. 세 단계를 표로 채워봐.

**ⓐ 시작 상태** (A→B→C→D→E, 프리 리스트 3→1→5, max=7)
```
head = 2, deleted = 3
 idx | data | next | dnext
  0  |  C   |  7   |  -
  1  |  -   |  -   |  5      ← 프리
  2  |  A   |  6   |  -
  3  |  -   |  -   |  1      ← 프리 (맨 앞)
  4  |  E   |  -1  |  -
  5  |  -   |  -   |  -1     ← 프리 (맨 끝)
  6  |  B   |  0   |  -
  7  |  D   |  4   |  -
```

**ⓑ 노드 F를 꼬리에 삽입**
- 교재: **"노드를 저장하는 곳은 프리 리스트의 머리 노드입니다."** → 레코드 ①____ 에 저장
- 프리 리스트는 ②________ 가 됨
- `max` 는? ③____ (교재: **"max값은 7로 유지됩니다"**) — 왜 안 늘어나?
- 꼬리였던 `E`(레코드 4)의 `next` 는 ④____ 로 바뀜

**ⓒ 노드 D를 삭제**
- `D`는 레코드 ⑤____ 에 있었어
- 그 레코드가 프리 리스트 **맨 앞**에 등록 → 프리 리스트는 ⑥________________
- `D`의 앞쪽 노드였던 `C`(레코드 0)의 `next` 는 ⑦____ 로 바뀜

🔥 **관찰 포인트**: 삽입할 때 `max` 가 **안 늘어났지?** 교재 352p:
> **"프리 리스트에 빈 레코드가 등록된 경우에는 '사용하지 않는 레코드(max번째 레코드 이후의 레코드)를 구하여 max를 증가시키고, 그 위치에 데이터를 저장'하지 않습니다."**

*(표를 채운 뒤 실행)*

In [ ]:
# 교재 그림 8-19 ⓐ 상태를 직접 구성
g = ArrayLinkedList(9)
g.n = [Node() for _ in range(9)]
g.n[0] = Node('C', 7); g.n[2] = Node('A', 6); g.n[4] = Node('E', -1)
g.n[6] = Node('B', 0); g.n[7] = Node('D', 4)
g.n[3].dnext = 1; g.n[1].dnext = 5; g.n[5].dnext = -1
g.head = 2; g.deleted = 3; g.max = 7; g.no = 5; g.current = 2

print("ⓐ 시작 상태"); show(g, upto=9)
g.add_last('F')
print("ⓑ 노드 F를 꼬리에 삽입"); show(g, upto=9)
print(f"   → max는 {g.max} 로 유지! 프리 리스트에서 재사용했으니까 🔥\n")
g.remove(7)
print("ⓒ 노드 D(레코드 7) 삭제"); show(g, upto=9)

### 10. 🟡 [설명] `get_insert_index` 의 3갈래

```python
def get_insert_index(self):
    if self.deleted == Null:              # 갈래 A
        if self.max + 1 < self.capacity:  # 갈래 A-1
            self.max += 1
            return self.max
        else:                             # 갈래 A-2
            return Null
    else:                                 # 갈래 B
        rec = self.deleted
        self.deleted = self.n[rec].dnext
        return rec
```

**표를 채워봐:**

| 갈래 | 조건 | 하는 일 | 반환값 |
|---|---|---|---|
| A-1 | 프리 리스트가 ①____ 이고 배열에 ②____ | ③ | ④ |
| A-2 | 프리 리스트가 비었고 배열이 ⑤____ | ⑥ | ⑦ |
| B | 프리 리스트에 ⑧____ | ⑨ | ⑩ |

- **우선순위가 왜 이 순서일까?** 재사용(B)을 먼저 검사하는 이유는? 🔥
- `max + 1 < capacity` 에서 왜 `<=` 가 아니라 `<` 야? capacity=100이면 최대 몇 개까지 쓸 수 있어?
- 🔥 **갈래 A-2로 가면 `Null` 을 반환하는데, 호출한 쪽은 이걸 어떻게 처리해?**
  `add_first` 를 보면 `if rec != Null:` 로 감싸져 있지. 그럼 **capacity를 초과하면** 어떻게 될까? 에러가 날까?

*(예측을 적은 뒤 실행)*

In [ ]:
print("[capacity 를 초과하면?]")
s = ArrayLinkedList(3)
for v in [1, 2, 3, 4, 5]:
    before = s.no
    s.add_last(v)
    print(f"  add_last({v}) → no: {before} → {s.no}  {'✅' if s.no > before else '❌ 무시됨!'}")
show(s, upto=3)
print("🔥 에러가 안 난다! 조용히 무시된다")
print("   → 호출한 쪽에서 len()이 늘었는지 확인하지 않으면 데이터 유실을 모른다\n")

print("[max + 1 < capacity 의 경계]")
for cap in (3, 5):
    t = ArrayLinkedList(cap)
    cnt = 0
    while True:
        before = t.no
        t.add_last('x')
        if t.no == before: break
        cnt += 1
    print(f"  capacity={cap} → 실제로 {cnt}개 저장 가능 (max는 {t.max}까지)")
print("  → max는 0부터 시작하므로 capacity개 전부 사용 가능 ✅")

### 11. 🔴 [설명] `delete_index` 의 2갈래

```python
def delete_index(self, idx):
    if self.deleted == Null:          # 갈래 A: 프리 리스트가 비어 있음
        self.deleted = idx
        self.n[idx].dnext = Null
    else:                             # 갈래 B: 프리 리스트에 이미 뭔가 있음
        rec = self.deleted
        self.deleted = idx
        self.n[idx].dnext = rec
```

- 갈래 A와 B가 하는 일을 각각 한 문장으로:
  - A: ①________________
  - B: ②________________
- 🔥 **사실 두 갈래를 하나로 합칠 수 있어.** 어떻게? 힌트: 갈래 A에서 `Null` 을 대입하는 건 사실 갈래 B의 `rec` 가 `Null` 인 경우와 같잖아?
  ```python
  def delete_index(self, idx):
      self.n[idx].dnext = ___     # ③
      self.deleted = ___          # ④
  ```
- 그럼 교재는 왜 두 갈래로 나눴을까? (의도 표현? 다른 언어의 관습?)
- 29일차 `add_first` 에서도 비슷한 게 있었지 — 빈 리스트도 분기 없이 자연스럽게 처리됐어. **`Null`(또는 `None`)을 "끝"으로 정한 설계**가 여기서도 분기를 없애줘.

*(답을 적은 뒤 실행)*

In [ ]:
class SimpleDelete(ArrayLinkedList):
    """delete_index를 2줄로 합친 버전"""
    def delete_index(self, idx: int) -> None:
        self.n[idx].dnext = self.deleted    # 기존 머리를 뒤에 붙이고
        self.deleted = idx                  # 자신이 새 머리가 된다

import random
random.seed(0)
print("두 버전이 항상 같은 결과를 내는가?\n")
def scenario(cls, seed):
    random.seed(seed)
    l = cls(20)
    log = []
    for _ in range(60):
        op = random.choice(['af','al','rf','rl'])
        if op=='af': l.add_first(random.randint(0,9))
        elif op=='al': l.add_last(random.randint(0,9))
        elif op=='rf': l.remove_first()
        else: l.remove_last()
        order, p = [], l.head
        while p != Null: order.append(l.n[p].data); p = l.n[p].next
        free, q = [], l.deleted
        while q != Null: free.append(q); q = l.n[q].dnext
        log.append((tuple(order), tuple(free), l.max, l.no))
    return log

diff = 0
for s in range(300):
    if scenario(ArrayLinkedList, s) != scenario(SimpleDelete, s): diff += 1
print(f"  랜덤 시나리오 300개 비교 → 결과가 다른 경우: {diff}회")
print("  → 완전히 동일! 2줄로 합쳐도 된다 ✅")
print("\n  교재가 2갈래로 나눈 건 '프리 리스트가 비었을 때'와 '아닐 때'를")
print("  명시적으로 드러내려는 의도로 보인다 (C 계열 언어의 관습이기도 함)")

---
# 💻 PART 4 — 코드 구현 (12~16번)

> R-1의 번역표를 손에 익히는 파트야. **어제 코드를 커서 버전으로 옮긴다.**

### 12. 🟢 [빈칸] `Node` 와 `__init__`

**기대 출력**
```
Node: data=-1, next=-1, dnext=-1
빈 리스트: head=-1 current=-1 max=-1 deleted=-1 no=0 capacity=6
len() = 0
```

In [ ]:
MyNull = -1

class MyNode:
    """연결 리스트용 노드 클래스(배열 커서 버전)"""
    def __init__(self, data=MyNull, next=MyNull, dnext=MyNull):
        self.data = ___          # ① 데이터
        self.next = ___          # ② 리스트의 뒤쪽 커서
        self.dnext = ___         # ③ 프리 리스트의 뒤쪽 커서


class MyArrayLinkedList:
    def __init__(self, capacity: int):
        self.head = ___          # ④ 머리 노드 (아직 없음)
        self.current = ___       # ⑤
        self.max = ___           # ⑥ 아직 아무 레코드도 안 씀
        self.deleted = ___       # ⑦ 프리 리스트도 비어 있음
        self.capacity = capacity
        self.n = [MyNode() for _ in range(capacity)]   # (17번 참고)
        self.no = 0

    def __len__(self) -> int:
        return ___               # ⑧


nd = MyNode()
print(f"Node: data={nd.data}, next={nd.next}, dnext={nd.dnext}")
l = MyArrayLinkedList(6)
print(f"빈 리스트: head={l.head} current={l.current} max={l.max} "
      f"deleted={l.deleted} no={l.no} capacity={l.capacity}")
print(f"len() = {len(l)}")

### 13. 🔴 [빈칸] 프리 리스트 관리 — 오늘의 핵심

8~11번에서 파헤친 그거야.

**기대 출력**
```
새 레코드 3개: 0 1 2
프리 등록 1, 2 → 프리 리스트: 2 → 1
재사용: 2 1 (LIFO)
그다음: 3 (새 레코드)
```

In [ ]:
def my_get_insert_index(self):
    """다음에 삽입할 레코드의 인덱스를 구함"""
    if ___:                              # ① 프리 리스트가 비었나?
        if self.max + 1 < self.capacity:
            self.max += ___              # ②
            return ___                   # ③ 새 레코드
        else:
            return ___                   # ④ 크기 초과
    else:
        rec = ___                        # ⑤ 프리 리스트 맨 앞
        self.deleted = ___               # ⑥ 그다음을 새 머리로
        return rec


def my_delete_index(self, idx: int) -> None:
    """레코드 idx를 프리 리스트에 등록"""
    if self.deleted == MyNull:
        self.deleted = ___               # ⑦
        self.n[idx].dnext = ___          # ⑧
    else:
        rec = self.deleted
        self.deleted = ___               # ⑨ idx가 새 머리
        self.n[idx].dnext = ___          # ⑩ 옛 머리를 뒤에


MyArrayLinkedList.get_insert_index = my_get_insert_index
MyArrayLinkedList.delete_index = my_delete_index

l = MyArrayLinkedList(6)
print("새 레코드 3개:", l.get_insert_index(), l.get_insert_index(), l.get_insert_index())
l.delete_index(1); l.delete_index(2)
free, p = [], l.deleted
while p != MyNull: free.append(str(p)); p = l.n[p].dnext
print(f"프리 등록 1, 2 → 프리 리스트: {' → '.join(free)}")
print("재사용:", l.get_insert_index(), l.get_insert_index(), "(LIFO)")
print("그다음:", l.get_insert_index(), "(새 레코드)")

### 14. 🟡 [빈칸] `search` 와 삽입 함수

R-1 번역표를 그대로 적용하면 돼.

**기대 출력**
```
search('C') = 2, current 레코드 = 2
search('Z') = -1
리스트: X → A → B → C → Y
```

In [ ]:
def my_search(self, data: Any) -> int:
    cnt = 0
    ptr = ___                            # ① 어디서 출발
    while ___:                           # ② 언제까지 (None이 아님!)
        if ___:                          # ③ 값 비교 (n[ptr] 사용)
            self.current = ptr
            return cnt
        cnt += 1
        ptr = ___                        # ④ 한 칸 뒤로
    return ___                           # ⑤ 실패


def my_add_first(self, data: Any) -> None:
    ptr = self.head                      # 삽입 전의 머리 레코드
    rec = ___                            # ⑥ 저장할 레코드 번호를 얻는다
    if rec != MyNull:
        self.head = self.current = ___   # ⑦
        self.n[self.head] = MyNode(data, ___)   # ⑧ 새 노드의 next
        self.no += 1


def my_add_last(self, data: Any) -> None:
    if ___:                              # ⑨ 리스트가 비었으면
        self.add_first(data)
    else:
        ptr = self.head
        while ___:                       # ⑩ 꼬리 레코드 찾기
            ptr = self.n[ptr].next
        rec = self.get_insert_index()
        if rec != MyNull:
            self.n[ptr].next = self.current = ___   # ⑪ 옛 꼬리가 새 노드를 가리킴
            self.n[rec] = MyNode(data)              # next 기본값 = Null ✅
            self.no += 1


MyArrayLinkedList.search = my_search
MyArrayLinkedList.add_first = my_add_first
MyArrayLinkedList.add_last = my_add_last

def order_of(l):
    r, p = [], l.head
    while p != MyNull: r.append(l.n[p].data); p = l.n[p].next
    return r

l = MyArrayLinkedList(8)
for v in ['A','B','C']: l.add_last(v)
print(f"search('C') = {l.search('C')}, current 레코드 = {l.current}")
print(f"search('Z') = {l.search('Z')}")
l.add_first('X'); l.add_last('Y')
print("리스트:", ' → '.join(order_of(l)))

### 15. 🔴 [빈칸] 삭제 함수 — 프리 리스트 등록을 잊지 마

29일차와의 **결정적 차이**: 삭제할 때 `delete_index` 를 **반드시** 불러야 해.
안 부르면 그 레코드가 **영원히 사용 불가**가 되거든 (메모리 누수!).

**기대 출력**
```
remove_first: ['B', 'C', 'D'] 프리=[0]
remove_last : ['B', 'C'] 프리=[3, 0]
remove(2)   : ['B'] 프리=[2, 3, 0]
빈 리스트 안전: [] no=0
```

In [ ]:
def my_remove_first(self) -> None:
    if self.head != MyNull:
        ptr = ___                        # ① 새 머리가 될 레코드를 먼저 기억!
        self.delete_index(___)           # ② 옛 머리를 프리 리스트로
        self.head = self.current = ___   # ③
        self.no -= 1


def my_remove_last(self) -> None:
    if self.head != MyNull:
        if ___:                          # ④ 노드가 1개뿐이면
            self.remove_first()
        else:
            ptr = self.head
            pre = self.head
            while ___:                   # ⑤ 꼬리 찾기
                pre = ptr
                ptr = ___                # ⑥
            self.n[pre].next = ___       # ⑦ 꼬리로의 커서를 끊는다
            self.delete_index(___)       # ⑧ 꼬리를 프리 리스트로
            self.current = pre
            self.no -= 1


def my_remove(self, p: int) -> None:
    if self.head != MyNull:
        if ___:                          # ⑨ p가 머리이면
            self.remove_first()
        else:
            ptr = self.head
            while ___:                   # ⑩ p의 앞쪽 레코드 찾기
                ptr = self.n[ptr].next
                if ptr == MyNull:
                    return
            self.delete_index(___)       # ⑪
            self.n[ptr].next = ___       # ⑫ 앞쪽이 p의 뒤쪽을 가리키게
            self.current = ptr
            self.no -= 1


MyArrayLinkedList.remove_first = my_remove_first
MyArrayLinkedList.remove_last = my_remove_last
MyArrayLinkedList.remove = my_remove

def free_of(l):
    r, p = [], l.deleted
    while p != MyNull: r.append(p); p = l.n[p].dnext
    return r

l = MyArrayLinkedList(8)
for v in ['A','B','C','D']: l.add_last(v)
l.remove_first(); print(f"remove_first: {order_of(l)} 프리={free_of(l)}")
l.remove_last();  print(f"remove_last : {order_of(l)} 프리={free_of(l)}")
l.remove(2);      print(f"remove(2)   : {order_of(l)} 프리={free_of(l)}")
e = MyArrayLinkedList(4); e.remove_first(); e.remove_last()
print(f"빈 리스트 안전: {order_of(e)} no={e.no}")

### 16. 🟡 [빈칸] `clear`, `next`, 이터레이터

**기대 출력**
```
next(): A → B → C, 마지막은 False
clear 후: no=0, head=-1, 프리=[2, 1, 0]
스캔: A B C
```

🔥 `clear()` 를 잘 봐. 29일차와 달리 **`self.no = 0` 이 없어.** 그래도 되는 이유는?

In [ ]:
def my_clear(self) -> None:
    while ___:                           # ① 리스트가 빌 때까지
        self.remove_first()
    self.current = ___                   # ②
    # 🔥 self.no = 0 이 없다! (아래 실행 결과를 보고 왜인지 설명해봐)


def my_next(self) -> bool:
    if ___ or ___:                       # ③④ 이동 불가한 두 경우
        return False
    self.current = ___                   # ⑤
    return True


class MyIterator:
    def __init__(self, n, head: int):
        self.n = n
        self.current = ___               # ⑥
    def __iter__(self):
        return self
    def __next__(self) -> Any:
        if ___:                          # ⑦
            raise StopIteration
        data = ___                       # ⑧ 지금 레코드의 데이터
        self.current = ___               # ⑨ 다음 레코드로
        return data


MyArrayLinkedList.clear = my_clear
MyArrayLinkedList.next = my_next
MyArrayLinkedList.__iter__ = lambda self: MyIterator(self.n, self.head)

l = MyArrayLinkedList(8)
for v in ['A','B','C']: l.add_last(v)
l.search('A')
path = [l.n[l.current].data]
while l.next(): path.append(l.n[l.current].data)
print(f"next(): {' → '.join(path)}, 마지막은 {l.next()}")

l2 = MyArrayLinkedList(8)
for v in ['A','B','C']: l2.add_last(v)
l2.clear()
print(f"clear 후: no={l2.no}, head={l2.head}, 프리={free_of(l2)}")

l3 = MyArrayLinkedList(8)
for v in ['A','B','C']: l3.add_last(v)
print("스캔:", *[e for e in l3])

---
# 🐛 PART 5 — 함정과 성능 (17~20번)

### 17. 🔴 [디버깅] 🔥 `[Node()] * capacity` 의 함정

```python
self.n = [Node()] * self.capacity
```

이 한 줄에 파이썬의 **고전적인 함정**이 숨어 있어.

- `[Node()] * 5` 는 `Node()` 를 **몇 번** 호출할까? ①____
- 그럼 `n[0]`, `n[1]`, ... `n[4]` 는 **각각 다른 객체**야, **같은 객체**야? ②____
- 확인식: `n[0] is n[1]` → ③____

**그런데 이 코드는 잘 동작해.** 왜일까? 🔥
- `add_first` 를 봐: `self.n[self.head] = Node(data, ptr)` — 새 객체를 **통째로 대입**해.
- `delete_index` 를 봐: `self.n[idx].dnext = Null` — 필드를 **수정**해. 그런데 `idx` 는 항상 **이미 대입된 슬롯**이야.
- 즉 **공유 중인 슬롯은 절대 수정되지 않아서** 문제가 안 드러나.

**💣 그럼 언제 터질까?** 아래 셀에서 확인해봐.
- 22일차 14번(`while j > 0`), 28일차 12번(`pt` 초기화), 29일차 20번(`no -= 1`)에 이어 또 하나의 **"우연히 안전한 코드"** 야.
- 고치는 법: ④________________

*(예측을 적은 뒤 실행)*

In [ ]:
print("[공유 여부 확인]")
t = ArrayLinkedList(5)
print(f"  n[0] is n[1] → {t.n[0] is t.n[1]}")
print(f"  모든 슬롯이 같은 객체? {all(t.n[0] is x for x in t.n)}")
print(f"  서로 다른 객체 개수: {len(set(id(x) for x in t.n))}개  ← 5개가 아니라 1개! 🔥\n")

print("[💣 언제 터지나 — 미사용 슬롯을 수정하면]")
t.n[3].data = '오염'
print(f"  n[3].data = '오염' 실행 후")
print(f"    n[0].data = {t.n[0].data!r}")
print(f"    n[4].data = {t.n[4].data!r}")
print("  → 하나만 바꿨는데 전부 바뀐다!\n")

print("[왜 교재 코드는 안 터지나]")
print("  · add_first/add_last: self.n[rec] = Node(...)  ← 슬롯을 '대입'으로 교체")
print("  · delete_index: self.n[idx].dnext = ...        ← 이미 교체된 슬롯만 '수정'")
print("  → 공유 중인 원본 Node는 한 번도 수정되지 않는다 (우연히 안전)\n")

print("[고친 버전]")
class FixedALL(ArrayLinkedList):
    def __init__(self, capacity):
        super().__init__(capacity)
        self.n = [Node() for _ in range(capacity)]   # ✅ 리스트 내포
f = FixedALL(5)
print(f"  서로 다른 객체 개수: {len(set(id(x) for x in f.n))}개 ✅")
f.n[3].data = '오염'
print(f"  n[3]만 바꾸면 n[0].data = {f.n[0].data!r}  ← 영향 없음 ✅")

### 18. 🟡 [디버깅] `remove()` 안의 죽은 코드

```python
def remove(self, p):
    ...
    else:
        ptr = self.head
        while self.n[ptr].next != p:
            ptr = self.n[ptr].next
            if ptr == Null:
                return
        self.n[ptr].next = Null              # ← ①
        self.delete_index(p)
        self.n[ptr].next = self.n[p].next    # ← ②
        self.current = ptr
        self.no -= 1
```

①에서 `self.n[ptr].next` 에 `Null` 을 넣었다가, ②에서 곧바로 **다른 값으로 덮어써.**

- ①번 줄을 **지워도** 결과가 같을까? 예측해봐.
- 만약 같다면, 이런 줄을 뭐라고 불러? ①________________ (dead code)
- 🔥 그런데 **위험한 시나리오**가 있어. 만약 `delete_index(p)` 가 `self.n[p].next` 를 건드린다면? 그럼 ②번 줄이 잘못된 값을 읽게 돼.
  - 실제로 `delete_index` 는 `dnext` 만 건드리지 `next` 는 안 건드려. 그래서 안전해.
  - 하지만 **순서에 의존하는 코드**야. `delete_index` 를 나중에 수정하는 사람이 `next` 도 초기화하면? 💣
- **더 안전하게 쓰려면** 순서를 어떻게 바꿔야 할까? ②________________

*(예측을 적은 뒤 실행)*

In [ ]:
class NoDeadCode(ArrayLinkedList):
    def remove(self, p: int) -> None:
        if self.head != Null:
            if p == self.head:
                self.remove_first()
            else:
                ptr = self.head
                while self.n[ptr].next != p:
                    ptr = self.n[ptr].next
                    if ptr == Null: return
                # self.n[ptr].next = Null   ← 지웠다
                self.delete_index(p)
                self.n[ptr].next = self.n[p].next
                self.current = ptr
                self.no -= 1

class SafeOrder(ArrayLinkedList):
    """읽기를 먼저, 등록을 나중에"""
    def remove(self, p: int) -> None:
        if self.head != Null:
            if p == self.head:
                self.remove_first()
            else:
                ptr = self.head
                while self.n[ptr].next != p:
                    ptr = self.n[ptr].next
                    if ptr == Null: return
                self.n[ptr].next = self.n[p].next   # ✅ 먼저 잇고
                self.delete_index(p)                # ✅ 그다음 등록
                self.current = ptr
                self.no -= 1

import random
def scenario(cls, seed):
    random.seed(seed)
    l = cls(20); log = []
    for _ in range(50):
        op = random.choice(['af','al','rf','rl','rm'])
        if op=='af': l.add_first(random.randint(0,9))
        elif op=='al': l.add_last(random.randint(0,9))
        elif op=='rf': l.remove_first()
        elif op=='rl': l.remove_last()
        else:
            if l.head != Null: l.remove(random.randrange(0, max(1,l.max+1)))
        o, p = [], l.head
        while p != Null: o.append(l.n[p].data); p = l.n[p].next
        fr, q = [], l.deleted
        while q != Null: fr.append(q); q = l.n[q].dnext
        log.append((tuple(o), tuple(fr), l.no))
    return log

d1 = sum(scenario(ArrayLinkedList,s) != scenario(NoDeadCode,s) for s in range(300))
d2 = sum(scenario(ArrayLinkedList,s) != scenario(SafeOrder,s) for s in range(300))
print(f"랜덤 300 시나리오")
print(f"  교재판 vs ①번 줄 삭제 → 다른 경우 {d1}회")
print(f"  교재판 vs 순서 교체   → 다른 경우 {d2}회")
print("\n→ ①번 줄은 죽은 코드. 지워도 결과가 같다 ✅")
print("→ '먼저 잇고 나중에 등록'이 순서 의존성이 없어 더 안전하다")

### 19. 🟢 [설명] 보충수업 8-3 — 파이썬 논리 연산자

교재 355p:
> **"많은 프로그래밍 언어에서 논리 연산자는 평가 결과로 True나 False 등의 논릿값을 생성합니다. 그런데 파이썬의 논리 연산자는 다른 프로그래밍 언어와 전혀 다른 성격을 갖고 있습니다."**

| 식 | 파이썬의 동작 |
|---|---|
| `x and y` | x를 평가하여 **거짓**이면 그 값을 생성. 그렇지 않으면 y를 평가하여 그 값을 생성 |
| `x or y` | x를 평가하여 **참**이면 그 값을 생성. 그렇지 않으면 y를 평가하여 그 값을 생성 |
| `not x` | x가 참이면 False, 그렇지 않으면 True |

**예측해봐:**
- `5 or 3` → ①____
- `0 or 3` → ②____
- `5 and 3` → ③____
- `0 and 3` → ④____
- `not 5` → ⑤____

🔥 **그리고 단축 평가(short circuit evaluation)**
> **"and 연산자의 왼쪽 피연산자를 평가한 값이 거짓이면, 오른쪽 피연산자의 평가를 생략합니다."**

**오늘 코드에서 단축 평가가 실제로 안전을 지키는 곳이 있어:**
```python
def next(self) -> bool:
    if self.current == Null or self.n[self.current].next == Null:
        return False
```
- `current == Null` 이 **참**이면 오른쪽은 ⑥________________
- 만약 단축 평가가 없어서 오른쪽도 평가된다면? `self.n[-1]` 을 읽게 되는데, 2번에서 봤듯 이건 **에러 없이 엉뚱한 값**을 줘. 💣

*(예측을 적은 뒤 실행)*

In [ ]:
print("[논리 연산자가 '값'을 반환한다]")
for expr, val in [("5 or 3", 5 or 3), ("0 or 3", 0 or 3),
                  ("5 and 3", 5 and 3), ("0 and 3", 0 and 3),
                  ("'' or 'x'", '' or 'x'), ("not 5", not 5)]:
    print(f"  {expr:12s} = {val!r}")
print("  → True/False가 아니라 '마지막에 평가한 피연산자의 값'\n")

print("[단축 평가 확인]")
def loud(name, val):
    print(f"    ⚠️ {name} 평가됨!")
    return val
print("  False and loud('오른쪽', True):", False and loud('오른쪽', True))
print("  True  or  loud('오른쪽', True):", True or loud('오른쪽', True))
print("  True  and loud('오른쪽', True):", True and loud('오른쪽', True))

print("\n[next()에서 단축 평가가 지켜주는 것]")
e = ArrayLinkedList(5)
print(f"  빈 리스트에서 next() → {e.next()}   (에러 없음 ✅)")
print(f"  만약 오른쪽도 평가했다면: self.n[{Null}] 접근")
print(f"    → self.n[-1].next = {e.n[Null].next}  (마지막 슬롯을 읽는다 💣)")
print("\n  💡 파이썬은 리스트 범위를 벗어난 음수 인덱스도 에러를 안 낸다.")
print("     그래서 '검사 후 접근' 순서가 특히 중요하다 (2번과 같은 이야기)")

### 20. 🔴 [실험] 🔥 교재의 주장은 사실일까

R-2에서 예측한 걸 확인할 시간이야. 그런데 **놀라운 결과가 나와.**

교재 342p의 주장을 다시 읽어봐:
> **"노드를 삽입·삭제할 때마다 내부에서 노드용 인스턴스를 생성하고 소멸합니다. 이때 메모리를 확보하고 해제하는 데 쓰는 비용을 결코 무시할 수 없습니다."**

→ 커서 방식은 **배열을 미리 확보해두고 재사용**하니 이 비용이 없어야 해. 정말 그럴까?

**먼저 코드를 다시 봐** 🔥
```python
def add_first(self, data):
    ptr = self.head
    rec = self.get_insert_index()
    if rec != Null:
        self.head = self.current = rec
        self.n[self.head] = Node(data, ptr)   # ← 여기!
        self.no += 1
```

- `self.n[self.head] = Node(data, ptr)` 는 **새 객체를 만드는 걸까, 기존 슬롯을 재사용하는 걸까?** ①____
- 그럼 교재가 말한 **"인스턴스 생성 비용 절약"** 이 이 구현에서 실현되고 있어? ②____
- 진짜로 재사용하려면 어떻게 써야 할까?
  ```python
  nd = self.n[rec]
  nd.data = ___ ; nd.next = ___ ; nd.dnext = ___   # ③ 필드만 갈아끼우기
  ```

**표를 완성해봐:**

| | 08-2 포인터 (29일차) | 08-3 커서 (오늘) |
|---|---|---|
| 크기 제한 | ④ 없음 | ⑤ |
| `add_first` | O(1) | ⑥ |
| `add_last` | O(n) | ⑦ |
| i번째 접근 | O(n) | ⑧ |
| 빈 리스트 `remove_first` | ⑨ **버그 (no가 음수)** | ⑩ |

**최종 질문 3개**
1. 실측에서 커서 방식이 포인터 방식보다 **빠를까 느릴까?** 예측하고 확인해봐.
2. 커서 방식의 **가장 큰 대가**는 뭐야? (힌트: `capacity`)
3. 교재 342p: **"프로그램을 실행하면서 데이터 개수가 크게 변하지 않거나 데이터 최대 개수를 예측할 수 있는 경우라면"** — 이 조건이 왜 붙었을까?

*(예측을 적은 뒤 실행)*

In [ ]:
print("[🔥 29일차 버그가 오늘은 고쳐져 있다]")
e = ArrayLinkedList(5)
for _ in range(3): e.remove_first()
print(f"  빈 리스트에 remove_first 3회 → no = {e.no}, len = {len(e)} ✅")
print("  08-3: self.no -= 1 이 if 블록 '안'에 있다")
print("  08-2: 바깥에 있어서 no=-3, len()이 ValueError 💥\\n")

# --- 진짜로 슬롯을 재사용하는 버전 ---
class TrueReuse(ArrayLinkedList):
    """새 Node 객체를 만들지 않고 배열 슬롯의 필드만 갱신"""
    def __init__(self, capacity):
        super().__init__(capacity)
        self.n = [Node() for _ in range(capacity)]   # 17번의 수정도 반영
    def add_first(self, data):
        ptr = self.head
        rec = self.get_insert_index()
        if rec != Null:
            self.head = self.current = rec
            nd = self.n[rec]
            nd.data = data; nd.next = ptr; nd.dnext = Null   # 필드만 갈아끼움
            self.no += 1

class PtrNode:
    __slots__ = ('data', 'next')
    def __init__(self, d=None, n=None): self.data = d; self.next = n

print("[삽입만 반복]")
print("       N | 교재 커서 | 진짜 재사용 | 포인터(08-2)")
print("  " + "-"*48)
for N in (3000, 10000, 30000):
    t = time.perf_counter()
    c = ArrayLinkedList(N+10)
    for i in range(N): c.add_first(i)
    e1 = time.perf_counter()-t
    t = time.perf_counter()
    r = TrueReuse(N+10)
    for i in range(N): r.add_first(i)
    e2 = time.perf_counter()-t
    t = time.perf_counter()
    h = None
    for i in range(N): h = PtrNode(i, h)
    e3 = time.perf_counter()-t
    print(f"  {N:6d} | {e1*1000:7.1f}ms | {e2*1000:9.1f}ms | {e3*1000:9.1f}ms")

print("\\n🔥 커서 방식이 포인터 방식보다 '느리다'!")
print("   이유 1: 교재 구현이 매번 Node(data, ptr)로 새 객체를 만든다")
print("           → 교재가 말한 '인스턴스 생성 비용 절약'을 스스로 포기한 셈")
print("   이유 2: n[ptr].next 는 '인덱싱 + 속성접근' 2단계라 ptr.next 보다 느리다")
print("   이유 3: 파이썬은 객체 할당/회수가 이미 최적화되어 있어 절약분이 작다")
print("   → 교재의 주장은 malloc/free 비용이 큰 C 계열 언어 기준이다\\n")

print("[삽입·삭제를 반복하는 워크로드 (재사용이 유리한 조건)]")
M = 20000
t = time.perf_counter()
c = ArrayLinkedList(200)
for i in range(M):
    c.add_first(i)
    if i % 2: c.remove_first()
e1 = time.perf_counter()-t
t = time.perf_counter()
r = TrueReuse(200)
for i in range(M):
    r.add_first(i)
    if i % 2: r.remove_first()
e2 = time.perf_counter()-t
t = time.perf_counter()
h = None
for i in range(M):
    h = PtrNode(i, h)
    if i % 2: h = h.next
e3 = time.perf_counter()-t
print(f"  교재 커서 {e1*1000:.1f}ms | 진짜 재사용 {e2*1000:.1f}ms | 포인터 {e3*1000:.1f}ms")
print("  → 여기서는 '진짜 재사용'이 가장 빠르다 (객체를 한 번도 새로 안 만드니까)")
print("     capacity=200으로 20000번을 처리했다 = 프리 리스트가 제 몫을 했다 ✅\\n")

print("[크기 제한 — 커서 방식의 대가]")
s = ArrayLinkedList(3)
for v in range(5): s.add_last(v)
print(f"  capacity=3 에 5개 삽입 → 실제 저장 {len(s)}개")
print("  🔥 초과분은 에러 없이 조용히 유실된다")
print("  → 그래서 교재가 '데이터 최대 개수를 예측할 수 있는 경우'라는 조건을 달았다")

---
---

# ✅ 정답 & 해설

> ⚠️ **배열 표를 직접 채운 뒤에 내려와.** 특히 4·9번은 손으로 그려야 남아.

---

## 🔁 Remind

### R-1 번역표

| 08-2 | 08-3 |
|---|---|
| `ptr = self.head` | ① **동일** (`ptr` 이 객체 참조 → 정수 인덱스로 바뀔 뿐) |
| `ptr is None` | ② **`ptr == Null`** |
| `ptr = ptr.next` | ③ **`ptr = self.n[ptr].next`** |
| `ptr.data` | ④ **`self.n[ptr].data`** |
| `self.head = None` | ⑤ **`self.head = Null`** |
| `Node(data, ptr)` | ⑥ **`self.n[rec] = Node(data, ptr)`** (레코드 번호가 필요!) |

🔥 **왜 하필 -1인가**: 배열 인덱스는 `0` 이상이라 **0은 유효한 레코드 번호**야. `head = 0` 은 "0번 레코드가 머리"라는 정상 상태지. 그래서 "없음"은 인덱스로 절대 나올 수 없는 값이어야 하고, `-1` 이 그 역할을 해.

### R-2
- 약점 1: ① **노드마다 객체 오버헤드 → 배열 대비 메모리 17배** (29일차 22번)
- 약점 2: ② **`add_last` 가 매번 꼬리까지 훑어서 O(n), n개 누적이 O(n²)** (29일차 21번)

- 커서 방식은 **약점 1(메모리)을 겨냥**해. 배열을 미리 확보해두니까.
- **약점 2는 그대로**야. `add_last` 는 여전히 `while` 로 꼬리를 찾아. → 20번에서 확인

---

## 🎯 PART 1 해설

### 1. 참조 대신 인덱스
- ① **A** ② **B** ③ **C** ④ **D** ⑤ **E** ⑥ **F** ⑦ **끝(리스트의 꼬리)**

**`-1` 이 적합한 이유**: 배열 인덱스로 절대 나올 수 없는 값이라 "유효한 위치"와 헷갈릴 일이 없어.

🔥 **실측**: 배열 순서 `['D','A','E','C','B','F']` vs 리스트 순서 `A → B → C → D → E → F`. 완전히 달라.

---

### 2. `Null = -1`
- ① 어제의 **`None`**
- ② **리스트가 비어 있다**
- ③ **p가 꼬리 노드다 (뒤쪽 노드 없음)**
- ④ **프리 리스트가 비어 있다 (재사용할 레코드 없음)**

🔥 **파이썬 함정**: `n[-1]` 은 에러가 아니라 **마지막 원소**를 줘. 실측에서 `l.n[Null].data` 가 조용히 값을 반환했지.

**코드가 막는 방법**: `while ptr != Null:` 을 **먼저** 검사하고 통과해야 `self.n[ptr]` 에 접근해. **"검사 후 접근"** 순서가 생명이야. 19번의 단축 평가도 같은 맥락이고.

25일차 4번(도수 정렬에서 `f[-3]`), 29일차 4번과 같은 계열의 함정이야.

---

### 3. 필드가 6개로 늘어난 이유

| 필드 | 담는 것 |
|---|---|
| `head` | ① 머리 노드의 **레코드 번호** |
| `current` | ② 주목 노드의 **레코드 번호** |
| `no` | ③ 노드 개수 |
| `capacity` | ④ 배열 크기 (최대 저장 가능 개수) |
| `max` | ⑤ **지금까지 사용해본 가장 큰 레코드 번호** |
| `deleted` | ⑥ **프리 리스트의 머리 레코드 번호** |

- **`max` 가 필요한 이유**: 프리 리스트가 비었을 때 **다음 새 레코드를 어디에 만들지** 알아야 해. `max + 1` 이 그 답이지.
- 🔥 **`dnext` 가 따로 필요한 이유**: 하나의 레코드가 **두 리스트에 동시에 속할 수는 없지만**, `next` 는 데이터 리스트용으로 이미 쓰이는 중이야. 삭제된 레코드의 `next` 에는 **옛날 값이 남아 있어서** 덮어쓰면 안 되고(18번!), 프리 리스트용 커서는 별도 필드가 필요해.

---

## 📊 PART 2 해설

### 4. 삽입 과정 — 그림 8-17
- ① `ptr = 1` ② `rec = 6` (프리 리스트가 비었으니 `max+1 = 6`)
- ③ `head = 6` ④ `n[6] = Node('G', 1)`
- ⑤ `head = 6` ⑥ `G` ⑦ `1` ⑧ `max: 5 → 6`

🔥 **기존 원소가 하나도 안 움직였어.** 29일차 1번에서 배열 삽입은 뒤의 원소를 전부 밀었는데, 여기선 **커서만 바꾸면 되니까** 이동이 0이야.

> 🔑 **"배열인데 밀리지 않는다"** — 물리적 위치와 논리적 순서를 분리했기 때문이야. 이게 커서 방식의 전부지.

---

### 5. `add_last` 추적

| 호출 | head | max | 배열 |
|---|---|---|---|
| 시작 | -1 | -1 | (비어 있음) |
| `add_last('A')` | ① **0** | ② **0** | ③ `0: A/-1` |
| `add_last('B')` | ④ **0** | ⑤ **1** | ⑥ `0: A/1, 1: B/-1` |
| `add_last('C')` | ⑦ **0** | ⑧ **2** | ⑨ `0: A/1, 1: B/2, 2: C/-1` |

- ⑩ `Node(data)` 의 `next` 기본값은 **`Null`**. 새 꼬리가 되니 정확히 맞는 값이야.
- ⑪ 꼬리 찾기 `while` 은 **O(n)**. 29일차 8번과 동일.
- **빈 리스트에 `add_first` 위임**: `head == Null` 인데 `self.n[ptr]` 을 하면 `n[-1]` 을 읽어 엉뚱한 동작을 해. 29일차는 `AttributeError` 로 시끄럽게 터졌는데, 오늘은 **조용히 틀린다**는 게 더 무섭지.

---

### 6. 🔥 배열 순서 ≠ 리스트 순서

- ① 0 ② 1 ③ 2 ④ **3** ⑤ **4**
- ⑥ 리스트: **Y → X → A → B → C**
- ⑦ 배열: **A, B, C, X, Y** (인덱스 0~4)

**`add_first` 인데 배열 뒤쪽에 저장되는 이유**: 저장 위치는 `get_insert_index()` 가 정하는데, 이 함수는 **리스트에서의 위치를 전혀 모르거든.** 그냥 "지금 비어 있는 레코드"를 줄 뿐이야. 리스트 순서는 **오직 `next` 커서**가 결정해.

> 🔑 교재 349p: **"연결 리스트에서 맨 끝이 아닌 것에 주의하세요."** — 이 한 문장이 오늘의 핵심이야.

---

## ♻️ PART 3 해설

### 7. 왜 프리 리스트인가
- ① **3개** (레코드 1, 3, 5)
- ② 방법 A: **O(capacity)** — 매번 배열 전체 스캔
- ③ 방법 B: **별도 리스트를 위한 추가 메모리가 필요하고, 그 리스트 자체를 또 관리해야 함**
- ④ 방법 C: **0 (추가 메모리 없음!)**

🔥 **방법 C가 천재적인 이유**: 빈 레코드는 어차피 데이터가 안 들어 있어. 그러니 그 안의 `dnext` 필드를 **공짜로 빌려** 서로 연결할 수 있지. 추가 배열이 전혀 필요 없어.

---

### 8. 🔥 프리 리스트는 스택이다
- ① **앞** ② **앞** ③ **LIFO (후입선출)** ④ **스택**

**실측**
```
레코드 1 삭제 → 프리: 1
레코드 3 삭제 → 프리: 3 → 1
레코드 5 삭제 → 프리: 5 → 3 → 1

재사용 순서: 5, 3, 1 (LIFO) → 다 쓰면 max+1 로 새 레코드
```

- ⑤ **5 → 3 → 1**
- ⑥⑦⑧ **5, 3, 1**

🔥 **교재 그림 8-19 ⓐ의 "3 → 1 → 5"는 삭제 순서가 5, 1, 3이었다는 뜻**이야. 삭제 순서와 프리 리스트 순서는 **정확히 역순**이지. (교재는 그 시점의 상태만 보여준 거고, 어떤 순서로 삭제했는지는 명시 안 했어)

> 🔑 4일차에 배운 스택이 여기서 다시 나와. **`push` = `delete_index`, `pop` = `get_insert_index`** 야.

---

### 9. 그림 8-19 추적
- ① **3** (프리 리스트 머리) ② **1 → 5** ③ **7 (유지)**
  - `max` 가 안 늘어나는 이유: **프리 리스트에서 재사용**했으니 새 레코드를 만들 필요가 없어.
- ④ **3** (F가 저장된 레코드)
- ⑤ **7** ⑥ **7 → 1 → 5** ⑦ **4** (D 다음이던 E의 레코드)

> 🔑 **삽입할 때 재사용을 우선하고, 없을 때만 배열을 늘린다.** 이 우선순위가 배열을 알뜰하게 쓰게 해줘.

---

### 10. `get_insert_index` 3갈래

| 갈래 | 조건 | 하는 일 | 반환 |
|---|---|---|---|
| A-1 | ① **비었고** ② **여유가 있음** | ③ `max += 1` | ④ **새 `max`** |
| A-2 | 비었고 ⑤ **가득 참** | ⑥ 아무것도 안 함 | ⑦ **`Null`** |
| B | ⑧ **재사용할 레코드가 있음** | ⑨ 프리 리스트 맨 앞을 떼어냄 | ⑩ **그 레코드 번호** |

- **재사용(B)을 먼저 검사하는 이유**: 배열을 아껴 쓰려고. 재사용 가능한데 새 레코드를 쓰면 배열이 금방 차버려.
- `max + 1 < capacity` 에서 `<` 인 이유: `max` 는 **0부터** 시작하는 인덱스야. `max+1` 이 곧 다음 인덱스인데, 그게 `capacity` 미만이어야 유효하지. 실측에서 capacity=3이면 정확히 **3개** 저장 가능(인덱스 0,1,2)함을 확인했어.
- 🔥 **capacity 초과 시**: `add_first/add_last` 가 `if rec != Null:` 로 감싸져 있어서 **에러 없이 조용히 무시**돼. `no` 도 안 늘어나. 호출한 쪽에서 `len()` 을 확인하지 않으면 **데이터가 유실된 걸 모른다.**

---

### 11. `delete_index` 2갈래
- A: ① **프리 리스트가 비어 있으니 idx가 유일한 원소가 된다 (dnext = Null)**
- B: ② **idx를 새 머리로 하고, 기존 머리를 그 뒤에 붙인다**

🔥 **2줄로 합칠 수 있어**:
```python
self.n[idx].dnext = self.deleted   # ③ 기존 머리를 뒤에 붙이고
self.deleted = idx                 # ④ 자신이 새 머리
```
`deleted` 가 `Null` 이면 `dnext = Null` 이 되어 갈래 A와 정확히 같아져.

**실측**: 랜덤 시나리오 300개 비교 → **결과가 다른 경우 0회**. 완전히 동일해.

- 교재가 나눈 이유: **"프리 리스트가 비었을 때"와 "아닐 때"를 명시적으로 드러내려는 의도**. C 계열 언어의 관습이기도 하고.
- 29일차 `add_first` 에서 빈 리스트도 분기 없이 처리됐던 것과 같은 이야기야. **`Null` 을 "끝"으로 정한 설계가 분기를 없애줘.**

---

## 💻 PART 4 해설

### 12~16. 빈칸 정답

**12번**
```python
self.data = data;  self.next = next;  self.dnext = dnext   # ①②③
self.head = MyNull; self.current = MyNull                  # ④⑤
self.max = MyNull;  self.deleted = MyNull                  # ⑥⑦
return self.no                                             # ⑧
```

**13번** (오늘의 핵심)
```python
if self.deleted == MyNull:              # ①
    self.max += 1                       # ②
    return self.max                     # ③
    return MyNull                       # ④
rec = self.deleted                      # ⑤
self.deleted = self.n[rec].dnext        # ⑥
# delete_index
self.deleted = idx                      # ⑦
self.n[idx].dnext = MyNull              # ⑧
self.deleted = idx                      # ⑨
self.n[idx].dnext = rec                 # ⑩
```

**14번**
```python
ptr = self.head                              # ①
while ptr != MyNull:                         # ②
    if self.n[ptr].data == data:             # ③
    ptr = self.n[ptr].next                   # ④
return MyNull                                # ⑤
rec = self.get_insert_index()                # ⑥
self.head = self.current = rec               # ⑦
self.n[self.head] = MyNode(data, ptr)        # ⑧
if self.head == MyNull:                      # ⑨
while self.n[ptr].next != MyNull:            # ⑩
self.n[ptr].next = self.current = rec        # ⑪
```

**15번** 🔥 **`delete_index` 호출을 잊으면 메모리 누수!**
```python
ptr = self.n[self.head].next     # ①  새 머리를 '먼저' 기억
self.delete_index(self.head)     # ②  그다음 옛 머리를 등록
self.head = self.current = ptr   # ③
if self.n[self.head].next == MyNull:   # ④
while self.n[ptr].next != MyNull:      # ⑤
    ptr = self.n[ptr].next             # ⑥
self.n[pre].next = MyNull              # ⑦
self.delete_index(ptr)                 # ⑧
if p == self.head:                     # ⑨  (커서는 정수라 == 사용!)
while self.n[ptr].next != p:           # ⑩
self.delete_index(p)                   # ⑪
self.n[ptr].next = self.n[p].next      # ⑫
```

⚠️ **⑨번 주의**: 29일차는 `p is self.head` 였는데 오늘은 **`p == self.head`** 야. 커서는 **정수**니까 `is` 로 비교하면 안 돼 (파이썬 작은 정수 캐싱 때문에 대부분 동작하지만 큰 값에서 깨져). 3일차 `is` vs `==` 의 실전 사례지.

⚠️ **①번 주의**: `delete_index` 를 부르기 **전에** 새 머리를 기억해둬야 해. 순서가 바뀌면 `n[head].next` 가 이미 프리 리스트에 등록된 뒤라 위험해.

**16번**
```python
while self.head != MyNull:                              # ①
self.current = MyNull                                   # ②
if self.current == MyNull or self.n[self.current].next == MyNull:  # ③④
self.current = self.n[self.current].next                # ⑤
self.current = head                                     # ⑥
if self.current == MyNull:                              # ⑦
data = self.n[self.current].data                        # ⑧
self.current = self.n[self.current].next                # ⑨
```

🔥 **`clear()` 에 `self.no = 0` 이 없어도 되는 이유**: `remove_first` 가 **`if` 블록 안에서** `no -= 1` 을 하니까, 반복하면 정확히 0에 도달해. **29일차에서는 이게 버그였는데 오늘은 제대로 되어 있어서** 안전장치가 필요 없는 거야. (20번)

---

## 🐛 PART 5 해설

### 17. 🔥 `[Node()] * capacity`
- ① **딱 1번**
- ② **같은 객체** ③ **True**

**실측**
```
n[0] is n[1] → True
서로 다른 객체 개수: 1개  ← 5개가 아니라!
n[3].data = '오염' → n[0].data도 '오염', n[4].data도 '오염'
```

**왜 안 터지나**: 코드가 슬롯을 **대입으로 교체**(`self.n[rec] = Node(...)`)하고, **수정**(`self.n[idx].dnext = ...`)은 이미 교체된 슬롯에만 해. 공유 중인 원본은 한 번도 안 건드려져.

- ④ 고치는 법: **`self.n = [Node() for _ in range(capacity)]`** — 리스트 내포로 매번 새 객체 생성

> 🔑 22일차 14번, 28일차 12번, 29일차 20번에 이어 **네 번째 "우연히 안전한 코드"** 야. 이번 달 내내 반복된 패턴이지: **동작하지만 이유가 코드 밖에 있으면 시한폭탄.**

---

### 18. 죽은 코드
- ① **죽은 코드(dead code)**
- **실측**: 랜덤 300 시나리오에서 ①번 줄을 지워도 **결과가 완전히 동일**

**위험한 이유**: `delete_index(p)` 가 `next` 를 안 건드린다는 **암묵적 가정**에 기대고 있어. 나중에 누가 `delete_index` 에 `self.n[idx].next = Null` 을 추가하면 ②번 줄이 잘못된 값을 읽게 돼. 💣

- ② **더 안전한 순서**: **먼저 잇고, 나중에 등록**
```python
self.n[ptr].next = self.n[p].next   # ✅ 먼저 리스트를 잇고
self.delete_index(p)                # ✅ 그다음 프리 리스트에 등록
```
실측에서 이 버전도 결과가 완전히 동일해. 그리고 **순서 의존성이 사라져서** 더 견고하지.

29일차 PART 3의 원칙 **"끊기 전에 먼저 이어라"** 가 여기서도 통해.

---

### 19. 파이썬 논리 연산자
- ① **5** ② **3** ③ **3** ④ **0** ⑤ **False**
- ⑥ **평가되지 않는다 (생략된다)**

**핵심**: 파이썬의 `and`/`or` 는 **True/False가 아니라 "마지막에 평가한 피연산자의 값"** 을 반환해.

**단축 평가가 안전을 지키는 곳**
```python
if self.current == Null or self.n[self.current].next == Null:
```
`current == Null` 이 참이면 오른쪽은 **평가 자체가 안 돼.** 만약 평가된다면 `self.n[-1].next` 를 읽는데, 2번에서 봤듯 **에러도 안 나고 엉뚱한 값**이 나와 💣

> 🔑 파이썬은 음수 인덱스를 허용하니까 **"검사 후 접근" 순서가 특히 중요해.** 다른 언어라면 인덱스 에러로 즉시 터졌을 텐데, 파이썬은 조용히 넘어가거든.

---

### 20. 🔥 교재의 주장은 사실일까

- ① **새 객체를 만든다** (`Node(data, ptr)`)
- ② **실현되지 않고 있다** 🔥
- ③ `nd.data = data; nd.next = ptr; nd.dnext = Null`

| | 08-2 포인터 | 08-3 커서 |
|---|---|---|
| 크기 제한 | ④ 없음 | ⑤ **`capacity` 로 고정** |
| `add_first` | O(1) | ⑥ **O(1)** |
| `add_last` | O(n) | ⑦ **O(n)** (그대로!) |
| i번째 접근 | O(n) | ⑧ **O(n)** (그대로!) |
| 빈 `remove_first` | ⑨ 버그 | ⑩ **정상 ✅** |

**실측 (삽입만 반복)**
```
       N | 교재 커서 | 진짜 재사용 | 포인터(08-2)
    3000 |    2.3ms |      1.4ms |       1.0ms
   10000 |    4.9ms |      4.9ms |       2.6ms
   30000 |   14.9ms |     20.5ms |       7.8ms
```

🔥 **커서 방식이 포인터 방식보다 느려!** 세 가지 이유:
1. 교재 구현이 매번 `Node(data, ptr)` 로 **새 객체를 만든다** — 절약 효과를 스스로 포기
2. `n[ptr].next` 는 **인덱싱 + 속성 접근 2단계**라 `ptr.next` 보다 느림
3. 파이썬은 객체 할당/회수가 이미 최적화되어 있어 절약분이 작음

**실측 (삽입·삭제 반복 — 재사용이 유리한 조건)**
```
교재 커서 7.9ms | 진짜 재사용 6.0ms | 포인터 7.2ms
```
여기서는 **진짜 재사용이 이겨.** capacity=200으로 20,000번을 처리했다는 게 핵심 — **프리 리스트가 제 몫을 한 거야.**

**최종 질문 답**
1. **파이썬에서는 대체로 느려.** 교재의 주장은 `malloc`/`free` 비용이 큰 **C 계열 언어** 기준이야.
2. **가장 큰 대가는 `capacity` 고정.** 초과하면 **에러도 없이 조용히 유실**돼. 실측에서 capacity=3에 5개 넣으니 3개만 저장됐지.
3. 배열을 미리 확보하는 방식이라, **최대 개수를 모르면** 너무 크게 잡아 메모리를 낭비하거나 너무 작게 잡아 데이터를 잃어. 그래서 그 조건이 붙은 거야.

> 🔑 **교재 코드를 그대로 믿지 말고 재봐라.** 23일차 4번(`sorted(a+b)` 가 더 빠름), 28일차 18번(BM 최악이 O(n)이 아님)에 이은 세 번째 "실측이 교재와 다른" 사례야.

---

## 📌 핵심 3줄 요약

1. **커서 = 참조 대신 배열 인덱스.** `None` 이 `Null(-1)` 로, `ptr.next` 가 `n[ptr].next` 로 바뀔 뿐 구조는 08-2와 1:1 대응이야. 대신 **배열의 물리적 위치와 리스트의 논리적 순서는 완전히 무관**해 — `add_first` 를 해도 배열 뒤쪽에 저장될 수 있어.
2. **프리 리스트는 배열 안의 두 번째 연결 리스트이자, 스택이다.** 삭제된 레코드의 `dnext` 필드를 공짜로 빌려 서로 잇고, 등록도 꺼내기도 맨 앞에서 하니 **LIFO**야. 덕분에 추가 메모리 0으로 빈 레코드를 관리해.
3. **교재의 "인스턴스 생성 비용 절약"은 이 구현에서 실현되지 않는다.** `self.n[rec] = Node(...)` 가 매번 새 객체를 만들거든. 필드만 갈아끼우도록 고쳐야 진짜 재사용이고, 그마저도 파이썬에서는 이득이 작아.

## 🗂️ 스터디 진행 가이드

- 🟢 **(R-1, 1, 2, 4, 5, 7, 12, 19번)**: 전원 필수
  - **R-1 번역표**를 손에 익히면 오늘 코드의 절반이 그냥 읽혀
  - **1번과 4번**은 종이에 표를 직접 채울 것
- 🟡 **(R-2, 3, 5, 9, 10, 14, 16, 18, 20번)**: 팀 목표선
  - **9번 그림 8-19 추적**이 프리 리스트 이해의 분수령
  - **20번 실측 반전**은 꼭 돌려볼 것
- 🔴 **(6, 8, 11, 13, 15, 17번)**: 도전
  - **8번(프리 리스트 = 스택)이 오늘 최대 수확** 🔥🔥 — 4일차 스택이 여기서 다시 나온다
  - **6번(배열 순서 ≠ 리스트 순서)** 을 못 잡으면 나머지가 다 흐려져
  - **13번**이 코드 중 가장 어려움. 8~11번 없이는 못 채워
  - **17번 `[Node()] * n`** 은 파이썬 개발자라면 평생 만나는 함정
- **금요일 코딩테스트 범위**: 🟢🟡 (R-1 ~ 20번)

## 🔗 오늘 회수된 개념들

- **29일차 전체** → 1:1 번역표 (R-1), 두 약점의 해결 여부 (R-2, 20번)
- **29일차 20번 `no -= 1` 버그** → 오늘은 `if` 안에 있어서 정상 (16, 20번)
- **4일차 스택(LIFO)** → 프리 리스트가 정확히 스택 (8번)
- **3일차 `is` vs `==`** → 커서는 정수라 `==` 를 써야 함 (15번)
- **25일차 4번 음수 인덱스 함정** → `n[-1]` 이 조용히 동작 (2, 19번)
- **22·28·29일차 "우연히 안전한 코드"** → `[Node()] * capacity` (17번)
- **23·28일차 "실측이 교재와 다름"** → 커서 방식이 오히려 느림 (20번)
- **29일차 "끊기 전에 먼저 이어라"** → `remove` 의 안전한 순서 (18번)

---

> **다음 진도 (31일차)**: 08-4 **원형 이중 연결 리스트**
>
> 오늘까지 남은 약점이 두 개 있어:
> - `add_last` 가 O(n) (꼬리를 매번 찾아야 함)
> - 앞쪽 노드로 되돌아갈 수 없음 (29일차 3번)
>
> **원형**(꼬리가 머리를 가리킴)이면 머리에서 한 칸 뒤로 가면 곧 꼬리 → `add_last` 가 **O(1)**
> **이중**(`prev` 커서 추가)이면 앞쪽으로도 갈 수 있음 → `remove_last` 에 `pre` 가 불필요
>
> 두 약점이 한 번에 해결돼. 08장의 완결편이야.